# DRIVE Retinal Vessel — Boundary Loss Ablation

비교: `ce_dice` / `plwce_dice` (baseline) vs `ce_dice_boundary` / `plwce_dice_boundary` / `plwce_boundary`

In [1]:
# === Cell 0: 환경 설정 ===
import subprocess, sys
for pkg in ['segmentation-models-pytorch', 'openpyxl', 'kagglehub', 'scipy', 'optuna', 'albumentations']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
import os, warnings, json, random, glob
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE'] = '1'
import numpy as np, cv2
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

DOMAIN = 'drive'; NUM_CLASSES = 2; CLASS_NAMES = ['Background', 'Vessel']
IMG_SIZE = 256; BATCH_SIZE = 4; NUM_WORKERS = 0; SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/drive'
os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}'); print('환경 설정 완료')

Device: cuda
환경 설정 완료


In [2]:
# === Cell 1: 데이터 로드 ===
import kagglehub
_path = kagglehub.dataset_download('andrewmvd/drive-digital-retinal-images-for-vessel-extraction')
print('DRIVE path:', _path)

all_tifs = sorted(glob.glob(os.path.join(_path, '**/*.tif'), recursive=True))
all_gifs = sorted(glob.glob(os.path.join(_path, '**/*.gif'), recursive=True))
train_img_paths  = sorted([p for p in all_tifs if 'training' in p.lower()])
train_mask_paths = sorted([p for p in all_gifs if 'training' in p.lower() and 'manual1' in p.lower()])
print(f'Train images: {len(train_img_paths)}  masks: {len(train_mask_paths)}')

tr_imgs, val_imgs, tr_masks, val_masks = train_test_split(
    train_img_paths, train_mask_paths, test_size=0.2, random_state=SEED)
print(f'Train: {len(tr_imgs)}  Val: {len(val_imgs)}')

def preprocess_drive(img_path):
    img = cv2.imread(img_path)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    green = clahe.apply(img[:, :, 1])
    rgb = cv2.merge([green, green, green])
    return cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))

train_tf = A.Compose([A.Normalize(), ToTensorV2()])
val_tf   = A.Compose([A.Normalize(), ToTensorV2()])

class DRIVEDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.imgs = img_paths; self.masks = mask_paths; self.augment = augment
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img  = preprocess_drive(self.imgs[i])
        mask = np.array(Image.open(self.masks[i]).convert('L'))
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.int64)
        if self.augment:
            if random.random() > 0.5: img = np.fliplr(img).copy(); mask = np.fliplr(mask).copy()
            if random.random() > 0.5: img = np.flipud(img).copy(); mask = np.flipud(mask).copy()
        t = train_tf(image=img)
        return t['image'], torch.from_numpy(mask)

train_loader = DataLoader(DRIVEDataset(tr_imgs, tr_masks, augment=True),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(DRIVEDataset(val_imgs, val_masks, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('클래스 비율 계산 중...')
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for mp in tr_masks:
    m = np.array(Image.open(mp).convert('L'))
    class_counts[1] += int((m > 127).sum())
    class_counts[0] += int((m <= 127).sum())
class_counts = class_counts.tolist()
total = sum(class_counts)
for c, (n, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  [{c}] {n:<12}: {cnt:>12,} ({100*cnt/total:.4f}%)')
print(f'BG:Vessel = {class_counts[0]/class_counts[1]:.1f}:1')
print(f'class_counts = {class_counts}')
print('DataLoader 완료')

Extracting files...
DRIVE path: /root/.cache/kagglehub/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/versions/1
Train images: 20  masks: 20
Train: 16  Val: 4
클래스 비율 계산 중...
  [0] Background  :    4,828,573 (91.4613%)
  [1] Vessel      :      450,787 (8.5387%)
BG:Vessel = 10.7:1
class_counts = [4828573, 450787]
DataLoader 완료


In [3]:
# === Cell 2: 모델 정의 ===

def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

class IterNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.out  = nn.Conv2d(64, 1, 1)
        self.iter = nn.Sequential(
            nn.Conv2d(65, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 1, 1))
    def forward(self, x):
        feat = self.base(x)
        out1 = self.out(feat)
        return [out1, self.iter(torch.cat([feat, out1], dim=1))]

def build_model():
    return IterNet().to(device)

def compute_val_dice(model, loader):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            res = preds[-1] if isinstance(preds, (list, tuple)) else preds
            prob = torch.sigmoid(res[:, 0])
            pred = (prob > 0.5).long()
            tp += ((pred == 1) & (masks == 1)).sum().item()
            fp += ((pred == 1) & (masks == 0)).sum().item()
            fn += ((pred == 0) & (masks == 1)).sum().item()
    return float(2 * tp / (2 * tp + fp + fn + 1e-8))

def compute_val_metrics(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            preds = model(imgs)
            res = preds[-1] if isinstance(preds, (list, tuple)) else preds
            prob = torch.sigmoid(res[:, 0]).cpu().numpy()
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())
    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    TP = ((all_preds == 1) & (all_labels == 1)).sum()
    FP = ((all_preds == 1) & (all_labels == 0)).sum()
    TN = ((all_preds == 0) & (all_labels == 0)).sum()
    FN = ((all_preds == 0) & (all_labels == 1)).sum()
    dice = 2 * TP / (2 * TP + FP + FN + 1e-8)
    sens = TP / (TP + FN + 1e-8)
    spec = TN / (TN + FP + 1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    return {'Dice': float(dice), 'Sensitivity': float(sens), 'Specificity': float(spec), 'AUC': float(auc)}

print('모델 + 유틸리티 함수 준비 완료')

모델 + 유틸리티 함수 준비 완료


In [4]:
# === Cell 3: 학습 함수 ===

FINAL_EPOCHS = 75
FINAL_LR     = 1e-4

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=75, lr=1e-4,
                subset_ratio=1.0, tag='', patience=15):
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    name = f'{loss_name}_a{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag: name = f'{tag}_{name}'
    print(f"\n{'='*60}\n{name}  (epochs={epochs})\n{'='*60}")
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset, random.sample(range(len(train_loader.dataset)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader
    history = {'loss': [], 'val_dice': []}
    best_dice = 0.0
    save_path = f'/tmp/ba_drive_{name}.pth'
    patience_counter = 0
    for epoch in range(epochs):
        # --- Boundary Loss annealing: 지연 선형 (20% warmup 후 선형, 3-comp: max=1/3, 2-comp: max=0.5) ---
        if criterion.boundary_loss is not None:
            WARMUP = int(epochs * 0.2)
            has_region = criterion.region_loss is not None
            alpha_max = 1/3 if has_region else 0.5
            if epoch <= WARMUP:
                alpha_t = 0.0
            else:
                alpha_t = min((epoch - WARMUP) / (epochs - WARMUP), 1.0) * alpha_max
            criterion.set_boundary_alpha(alpha_t)
        model.train()
        epoch_loss = 0.0
        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = 0
            for p in preds:
                loss = loss + criterion(to_2ch_logits(p), masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)
        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
            patience_counter = 0
        else:
            patience_counter += 1
        print()
        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1} (patience={patience})')
            break
    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice

print('train_model() 준비 완료')

train_model() 준비 완료


In [5]:
# === Cell 4: Optuna — PLWCE alpha 탐색 (Boundary Loss 환경) ===
import optuna, traceback
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW    = 2.5
ALPHA_HIGH   = 15.0
PROXY_EPOCHS = 8
PROXY_RATIO  = 0.15
N_TRIALS     = 20

def make_objective(loss_name):
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, dice = train_model(loss_name, alpha=alpha,
                                     epochs=PROXY_EPOCHS, subset_ratio=PROXY_RATIO)
            return dice
        except Exception:
            traceback.print_exc()
            return None
    return objective

# --- plwce_dice_boundary ---
print('Optuna: plwce_dice_boundary ...')
sampler_pdb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()}
)
study_pdb = optuna.create_study(direction='maximize', sampler=sampler_pdb)
study_pdb.optimize(make_objective('plwce_dice_boundary'), n_trials=N_TRIALS)
best_trials_pdb = [t for t in study_pdb.trials if t.value is not None]
best_alpha_with_dice = (
    best_trials_pdb[int(np.argmax([t.value for t in best_trials_pdb]))].params['alpha']
    if best_trials_pdb else 5.0
)
print(f'  best alpha (with Dice): {best_alpha_with_dice:.3f}')

# --- plwce_boundary (no Dice) ---
print('Optuna: plwce_boundary (no Dice) ...')
sampler_pb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()}
)
study_pb = optuna.create_study(direction='maximize', sampler=sampler_pb)
study_pb.optimize(make_objective('plwce_boundary'), n_trials=N_TRIALS)
best_trials_pb = [t for t in study_pb.trials if t.value is not None]
best_alpha_without_dice = (
    best_trials_pb[int(np.argmax([t.value for t in best_trials_pb]))].params['alpha']
    if best_trials_pb else 7.0
)
print(f'  best alpha (no Dice):   {best_alpha_without_dice:.3f}')

optuna_data = {
    'plwce_dice_boundary': {'best_alpha': best_alpha_with_dice},
    'plwce_boundary':      {'best_alpha': best_alpha_without_dice},
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json'), 'w') as f:
    json.dump(optuna_data, f, indent=2)
print('Optuna 결과 저장 완료')

Optuna: plwce_dice_boundary ...


[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)

plwce_dice_boundary_a14.34  (epochs=8)
Ep01 | Loss: 1.5503 | Val Dice: 0.1657  <- Best!
Ep02 | Loss: 1.4840 | Val Dice: 0.1657
Ep03 | Loss: 1.3784 | Val Dice: 0.1658  <- Best!
Ep04 | Loss: 1.2662 | Val Dice: 0.1666  <- Best!
Ep05 | Loss: 1.1929 | Val Dice: 0.1734  <- Best!
Ep06 | Loss: 1.1121 | Val Dice: 0.2182  <- Best!
Ep07 | Loss: 1.0552 | Val Dice: 0.2246  <- Best!
Ep08 | Loss: 0.9932 | Val Dice: 0.2315  <- Best!
최고 Val Dice: 0.2315
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)

plwce_dice_boundary_a3.16  (epochs=8)
Ep01 | Loss: 1.6563 | Val Dice: 0.1343  <- Best!
Ep02 | Loss: 1.6148 | Val Dice: 0.1192
Ep03 | Loss: 1.5251 | Val Dice: 0.1237
Ep04 | Loss: 1.4222 | Val Dice: 0.1344  <- Best!
Ep05 | Loss: 1.3297 | Val Dice: 0.1382  <- Best!
Ep06 | Loss: 1.2497 | Val Dice

In [6]:
# === Cell 5: Boundary Ablation 전체 학습 ===

# --- Optuna 결과 로드 (Cell 4 미실행 시 JSON fallback) ---
try:
    _ = best_alpha_with_dice
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json')) as f:
            d = json.load(f)
        best_alpha_with_dice    = d['plwce_dice_boundary']['best_alpha']
        best_alpha_without_dice = d['plwce_boundary']['best_alpha']
        print(f'Optuna 로드: with_dice={best_alpha_with_dice:.3f}, no_dice={best_alpha_without_dice:.3f}')
    except FileNotFoundError:
        best_alpha_with_dice    = 5.0
        best_alpha_without_dice = 7.0
        print(f'Optuna 미실행 → fallback alpha with_dice=5.0, no_dice=7.0')

experiments = [
    ('ce_dice',             1.0,                     'CE+Dice                     [baseline]'),
    ('plwce_dice',          best_alpha_with_dice,    f'PLWCE+Dice                  (α={best_alpha_with_dice:.3f}) [baseline]'),
    ('ce_dice_boundary',    1.0,                     'CE+Dice+BL                  [literature]'),
    ('plwce_dice_boundary', best_alpha_with_dice,    f'PLWCE+Dice+BL               (α={best_alpha_with_dice:.3f})'),
    ('plwce_boundary',      best_alpha_without_dice, f'PLWCE+BL     (no Dice)       (α={best_alpha_without_dice:.3f})'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_dice = train_model(
        loss_name=loss_name, alpha=alpha,
        epochs=FINAL_EPOCHS, lr=FINAL_LR, tag='ba')
    all_results[label] = {
        'model': model, 'history': history, 'best_dice': best_dice,
        'loss_name': loss_name, 'alpha': alpha,
    }

print('\n' + '='*65)
print('[Boundary Ablation 요약 — Val Dice]')
print(f"{'Loss':<52} {'Val Dice':>9}")
print('-'*63)
for label, v in all_results.items():
    print(f"{label:<52} {v['best_dice']:>9.4f}")

[ce_dice] Components: CE + DiceLoss  (λ=0.500 each)

ba_ce_dice  (epochs=75)


Ep01 | Loss: 1.4699 | Val Dice: 0.3182  <- Best!
Ep02 | Loss: 1.3652 | Val Dice: 0.2217
Ep03 | Loss: 1.3016 | Val Dice: 0.2708
Ep04 | Loss: 1.2545 | Val Dice: 0.3697  <- Best!
Ep05 | Loss: 1.2209 | Val Dice: 0.4522  <- Best!
Ep06 | Loss: 1.1920 | Val Dice: 0.5140  <- Best!
Ep07 | Loss: 1.1683 | Val Dice: 0.5511  <- Best!
Ep08 | Loss: 1.1410 | Val Dice: 0.5799  <- Best!
Ep09 | Loss: 1.1187 | Val Dice: 0.5941  <- Best!
Ep10 | Loss: 1.1075 | Val Dice: 0.6004  <- Best!
Ep11 | Loss: 1.0884 | Val Dice: 0.6040  <- Best!
Ep12 | Loss: 1.0711 | Val Dice: 0.6063  <- Best!
Ep13 | Loss: 1.0588 | Val Dice: 0.6099  <- Best!
Ep14 | Loss: 1.0466 | Val Dice: 0.6135  <- Best!
Ep15 | Loss: 1.0318 | Val Dice: 0.6094
Ep16 | Loss: 1.0249 | Val Dice: 0.6139  <- Best!
Ep17 | Loss: 1.0119 | Val Dice: 0.6273  <- Best!
Ep18 | Loss: 0.9998 | Val Dice: 0.6309  <- Best!
Ep19 | Loss: 0.9912 | Val Dice: 0.6310  <- Best!
Ep20 | Loss: 0.9842 | Val Dice: 0.6297
Ep21 | Loss: 0.9728 | Val Dice: 0.6367  <- Best!
Ep22 | Loss

In [7]:
# === Cell 6: 평가 및 결과 저장 ===
import pandas as pd

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for i, (label, v) in enumerate(all_results.items()):
    c = COLORS[i % len(COLORS)]
    ax1.plot(v['history']['loss'],     label=label[:40], color=c)
    ax2.plot(v['history']['val_dice'], label=label[:40], color=c)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)
ax2.set_title('Val Dice');      ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_curves.png'), dpi=150)
plt.show()

# --- Val set 정량 평가 (DRIVE test GT 없음) ---
print('\n[Val Set 정량 평가 (DRIVE test GT 없음)]')
print(f"{'Loss':<52} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 82)

final_results = {}
for label, v in all_results.items():
    m = compute_val_metrics(v['model'], val_loader)
    final_results[label] = m
    print(f"{label:<52} {m['Dice']:>7.4f} {m['Sensitivity']:>7.4f} {m['Specificity']:>7.4f} {m['AUC']:>7.4f}")

# --- 바 차트 ---
labels = list(final_results.keys())
dices  = [final_results[l]['Dice'] for l in labels]
idx    = sorted(range(len(dices)), key=lambda i: dices[i], reverse=True)
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(labels)), [dices[i] for i in idx],
              color=[COLORS[i % len(COLORS)] for i in range(len(labels))])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([labels[i][:40] for i in idx], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Val Dice')
ax.set_title(f'Boundary Ablation — Val Dice ({DOMAIN.upper()})')
for bar, val in zip(bars, [dices[i] for i in idx]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_bar.png'), dpi=150)
plt.show()

# --- JSON 저장 ---
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.json'), 'w') as f:
    json.dump({l: {k: v for k, v in m.items()} for l, m in final_results.items()},
              f, indent=2, ensure_ascii=False)
print(f'결과 저장 완료: {RESULTS_DIR}')

# --- Excel 저장 ---
summary_rows = []
for label, m in final_results.items():
    summary_rows.append({
        'Loss_Function': label,
        'Dice':          round(m['Dice'],        4),
        'Sensitivity':   round(m['Sensitivity'], 4),
        'Specificity':   round(m['Specificity'], 4),
        'AUC':           round(m['AUC'],         4),
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
            zip(v['history']['loss'], v['history']['val_dice']), 1):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')


[Val Set 정량 평가 (DRIVE test GT 없음)]
Loss                                                    Dice    Sens    Spec     AUC
----------------------------------------------------------------------------------
CE+Dice                     [baseline]                0.6695  0.7291  0.9556  0.9446
PLWCE+Dice                  (α=7.763) [baseline]      0.6622  0.7962  0.9398  0.9514
CE+Dice+BL                  [literature]              0.6683  0.7707  0.9470  0.9518
PLWCE+Dice+BL               (α=7.763)                 0.6618  0.7825  0.9424  0.9515
PLWCE+BL     (no Dice)       (α=7.763)                0.6704  0.7931  0.9433  0.9545
결과 저장 완료: /root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/drive
Excel 저장: /root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/drive/drive_boundary_ablation.xlsx
